
# Research Paper Walkthrough: *Attention Is All You Need* (Transformer) — Equation-by-Equation
## A “paper reading” notebook: shapes → meaning → code

Paper: **Vaswani et al., 2017** (NeurIPS) “Attention Is All You Need” (Transformer).

**Goal of this notebook**
- Make you able to **read the math in the paper fluently**
- Decode each key equation as:
  1) What objects are involved (scalar / vector / matrix / tensor)?
  2) What are the **shapes**?
  3) What does it mean **in plain English**?
  4) How do we implement it in **PyTorch**?

> Tip: Keep a small “shape ledger” next to you while reading papers. Most confusion is shape confusion.

---



## 0) Notation & Shapes we will use

We’ll use a **batch-first** convention (common in PyTorch code):

- Batch size: $B$
- Sequence length: $T$
- Model dimension: $d_{\text{model}}$
- Number of heads: $h$
- Head dimension: $d_k = d_v = d_{\text{model}}/h$ (in the paper's standard setting)

Typical tensors:
- Token embeddings / hidden states: $H \in \mathbb{R}^{B \times T \times d_{\text{model}}}$
- Projection matrices (per head often): $W^Q, W^K, W^V$
- For multi-head: we reshape into heads:
  - $Q, K, V \in \mathbb{R}^{B \times h \times T \times d_k}$

We'll implement a minimal transformer block from scratch.


In [ ]:

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)



# 1) Scaled Dot-Product Attention (Equation 1)

The paper defines:

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

### What are Q, K, V?
They are **matrices (or tensors)** derived from the same (self-attention) or different sources (cross-attention).

For a *single head* and a *single sequence* (no batch):
- $Q \in \mathbb{R}^{T \times d_k}$
- $K \in \mathbb{R}^{T \times d_k}$
- $V \in \mathbb{R}^{T \times d_v}$

### Shape-check the equation
1) $QK^\top$:
- $Q$ is $(T \times d_k)$
- $K^\top$ is $(d_k \times T)$
- So $QK^\top$ is $(T \times T)$

Interpretation: every query token compares to every key token → **all-to-all token similarity**.

2) Divide by $\sqrt{d_k}$:
Scaling keeps dot products from growing too large (helps softmax gradients).

3) softmax over last dimension:
Gives attention weights matrix $A \in \mathbb{R}^{T \times T}$ where each row sums to 1.

4) Multiply by $V$:
- $A$ is $(T \times T)$
- $V$ is $(T \times d_v)$
- Output is $(T \times d_v)$

### Plain English
For each token (each row):
- compute similarity to all tokens
- convert similarities into probabilities (weights)
- output = weighted average of value vectors

---


In [ ]:

def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q, K, V: (B, h, T, d_k)
    d_k = Q.size(-1)
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)  # (B,h,T,T)
    if mask is not None:
        # mask is broadcastable to (B,h,T,T), with False/0 meaning "block"
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)              # (B,h,T,T)
    out = weights @ V                                    # (B,h,T,d_k)
    return out, weights

B, h, T, d_k = 2, 4, 5, 8
Q = torch.randn(B, h, T, d_k)
K = torch.randn(B, h, T, d_k)
V = torch.randn(B, h, T, d_k)

out, w = scaled_dot_product_attention(Q, K, V)
out.shape, w.shape, w[0,0,0].sum().item()



### “Why divide by $\sqrt{d_k}$?” (intuition)
If components of Q and K are roughly unit-scale, the dot product magnitude grows with $d_k$.
Large logits push softmax into saturated regions → tiny gradients. Scaling stabilizes training.



# 2) Multi-Head Attention (Equation 2)

The paper defines:

$$
\mathrm{head}_i = \mathrm{Attention}(QW_i^Q, KW_i^K, VW_i^V)
$$

$$
\mathrm{MultiHead}(Q,K,V) = \mathrm{Concat}(\mathrm{head}_1,\dots,\mathrm{head}_h)W^O
$$

### Intuition
Instead of one attention “view” of the sequence, use $h$ different learned projections.
Each head can specialize (syntax, coreference, positional patterns, etc.).

### Shapes
Let input hidden states be:
- $H \in \mathbb{R}^{B \times T \times d_{\text{model}}}$

We compute:
- $Q = HW^Q$, $K = HW^K$, $V = HW^V$ with $W^Q,W^K,W^V \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$
- reshape into heads: $(B, h, T, d_k)$

After attention:
- concatenate heads back: $(B, T, h\cdot d_k) = (B,T,d_{\text{model}})$
- final projection with $W^O \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$

---


In [ ]:

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, H, mask=None):
        # H: (B,T,d_model)
        B, T, _ = H.shape

        Q = self.Wq(H)  # (B,T,d_model)
        K = self.Wk(H)
        V = self.Wv(H)

        # reshape to (B,h,T,d_k)
        Q = Q.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

        out, weights = scaled_dot_product_attention(Q, K, V, mask=mask)  # (B,h,T,d_k)

        # back to (B,T,d_model)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        out = self.Wo(out)
        return out, weights

B, T, d_model, h = 2, 6, 32, 4
H = torch.randn(B, T, d_model)
mha = MultiHeadSelfAttention(d_model, h)

out, weights = mha(H)
out.shape, weights.shape



# 3) Position-wise Feed-Forward Network (Equation 3)

The paper defines a 2-layer MLP applied **independently at each position**:

$$
\mathrm{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2
$$

(Modern variants often use GELU instead of ReLU, but the original uses ReLU.)

### Intuition
Attention mixes information across tokens.
FFN mixes information across **features** (channels) at each token.

### Shapes
For each position vector $x \in \mathbb{R}^{d_{\text{model}}}$:
- $W_1 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$
- $W_2 \in \mathbb{R}^{d_{\text{ff}} \times d_{\text{model}}}$

Applied to batch/sequence tensor $H \in \mathbb{R}^{B\times T\times d_{\text{model}}}$ as a pointwise MLP.

---


In [ ]:

class PositionwiseFFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

B, T, d_model, d_ff = 2, 6, 32, 64
ffn = PositionwiseFFN(d_model, d_ff)
H = torch.randn(B, T, d_model)
ffn(H).shape



# 4) Residual Connections + LayerNorm (sublayer wrapper)

The paper uses a pattern around each sublayer:

$$
\mathrm{LayerNorm}(x + \mathrm{Sublayer}(x))
$$

### Intuition
- Residual: helps gradients flow; the sublayer learns a “delta” to add.
- LayerNorm: normalizes features per token for stability.

### LayerNorm formula (feature-wise)
For a token vector $x \in \mathbb{R}^{d}$:
$$
\mu = \frac{1}{d}\sum_{i=1}^d x_i,\quad
\sigma^2 = \frac{1}{d}\sum_{i=1}^d (x_i-\mu)^2
$$

$$
\mathrm{LN}(x) = \gamma \odot \frac{x-\mu}{\sqrt{\sigma^2+\epsilon}} + \beta
$$

- $\gamma,\beta$ are learned scale/shift parameters (same shape as features)
- $\epsilon$ is a small constant for numerical stability

---


In [ ]:

# Demonstrate LayerNorm effect on one token vector
x = torch.randn(10)
ln = nn.LayerNorm(10)
y = ln(x)

x_mean, x_std = x.mean().item(), x.std(unbiased=False).item()
y_mean, y_std = y.mean().item(), y.std(unbiased=False).item()

(x_mean, x_std, y_mean, y_std)



# 5) Positional Encoding (sinusoidal)

Because the transformer has no recurrence/convolution, the paper adds position information.

They define sinusoidal positional encodings:
$$
\mathrm{PE}_{(pos,2i)}=\sin\left(pos/10000^{2i/d_{\text{model}}}\right)
$$
$$
\mathrm{PE}_{(pos,2i+1)}=\cos\left(pos/10000^{2i/d_{\text{model}}}\right)
$$

### Intuition
- Each dimension is a sinusoid at a different frequency
- Relative positions become learnable via linear combinations
- Allows extrapolation to longer sequences than seen in training (as claimed in paper)

---


In [ ]:

def sinusoidal_positional_encoding(T, d_model, device=None):
    device = device or "cpu"
    pe = torch.zeros(T, d_model, device=device)
    pos = torch.arange(T, device=device).float().unsqueeze(1)  # (T,1)
    i = torch.arange(d_model, device=device).float().unsqueeze(0)  # (1,d)
    # Compute the angles for even indices
    angle_rates = 1.0 / (10000 ** (2 * (i // 2) / d_model))
    angles = pos * angle_rates
    pe[:, 0::2] = torch.sin(angles[:, 0::2])
    pe[:, 1::2] = torch.cos(angles[:, 1::2])
    return pe  # (T,d_model)

T, d_model = 8, 16
pe = sinusoidal_positional_encoding(T, d_model)
pe.shape, pe[0, :6], pe[1, :6]



# 6) Masking (decoder “causal” self-attention)

In the decoder, token at position t must not attend to future tokens (> t).

We implement a causal mask $M\in\{0,1\}^{T\times T}$ where:
- $M_{ij}=1$ if $j \le i$
- $M_{ij}=0$ if $j > i$ (blocked)

In attention computation, we set blocked scores to $-\infty$ before softmax.

---


In [ ]:

def causal_mask(T, device=None):
    device = device or "cpu"
    # lower-triangular ones
    return torch.tril(torch.ones(T, T, device=device)).bool()

T = 6
mask = causal_mask(T)
mask.int()


In [ ]:

# Demonstrate masked attention weight rows sum to 1 but only over allowed positions
B, h, T, d_k = 1, 2, 6, 8
Q = torch.randn(B, h, T, d_k)
K = torch.randn(B, h, T, d_k)
V = torch.randn(B, h, T, d_k)

mask_bt = causal_mask(T).unsqueeze(0).unsqueeze(0)  # (1,1,T,T) broadcast
out, weights = scaled_dot_product_attention(Q, K, V, mask=mask_bt)

weights[0,0,0], weights[0,0,0].sum().item(), weights[0,0,0,1:].sum().item()



# 7) Build a minimal Transformer Encoder Block (paper structure)

An encoder block (conceptually) is:

1) Multi-head self-attention  
2) Add & LayerNorm  
3) Position-wise FFN  
4) Add & LayerNorm  

We’ll implement a simplified encoder block.

---


In [ ]:

class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.0):
        super().__init__()
        self.mha = MultiHeadSelfAttention(d_model, num_heads)
        self.ffn = PositionwiseFFN(d_model, d_ff)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: (B,T,d_model)
        attn_out, attn_w = self.mha(x, mask=mask)
        x = self.ln1(x + self.dropout(attn_out))

        ffn_out = self.ffn(x)
        x = self.ln2(x + self.dropout(ffn_out))
        return x, attn_w

B, T, d_model = 2, 8, 32
x = torch.randn(B, T, d_model)
block = TransformerEncoderBlock(d_model=d_model, num_heads=4, d_ff=64, dropout=0.0)
y, attn_w = block(x)

y.shape, attn_w.shape



# 8) Training Objective (what the paper optimizes)

For machine translation (seq2seq), training uses **maximum likelihood** / **cross-entropy**:

- Model outputs logits for each next token
- Apply softmax to get probabilities
- Loss = negative log probability of the true token

For each position t:
$$
\mathcal{L}_t = -\log p_\theta(y_t \mid y_{<t}, x)
$$

Total loss sums over positions (and batch).

Below is a tiny “next-token” loss demo.

---


In [ ]:

B, T, V = 2, 5, 100  # vocab size V
logits = torch.randn(B, T, V)     # model output
targets = torch.randint(0, V, (B, T))

loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))
loss.item()



# 9) How to decode any equation you see in the paper (a repeatable template)

When you see something like:

$$
\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

Write a little table:

| Symbol | Type | Shape | Meaning |
|---|---|---|---|
| Q | tensor | (B,h,T,d_k) | queries |
| K | tensor | (B,h,T,d_k) | keys |
| V | tensor | (B,h,T,d_k) | values |
| QK^T | tensor | (B,h,T,T) | similarities |
| softmax(...) | tensor | (B,h,T,T) | weights (rows sum to 1) |
| ... V | tensor | (B,h,T,d_k) | weighted sums |

Do this and you will not get lost.

---

## Next step (optional but powerful)
Pick one section of the paper and try to rewrite it in your own words and shapes.  
That is how you internalize it.
